# NimbusPay Ticket Triage — LLM Fine-Tuning Assessment

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Training:** PEFT LoRA / QLoRA, maximum 50M trainable parameters  
**Target runtime:** Google Colab free T4  
**Final inference:** exactly the supplied `system` + `user` messages, model chat template, greedy decoding, `max_new_tokens=200`.

> **Data handling:** Upload the original assessment ZIP when prompted. It is read only in this Colab runtime. Do not upload `candidate_pack/data/` or the base model to GitHub. Dev is for evaluation only; test is inference only. This notebook never trains on dev or test.

Run every cell from top to bottom on a fresh Colab T4 runtime. The experiment section performs five training runs so that the four required one-variable comparisons are covered. No result is assumed in advance; metrics are written from actual runs.

## 0. Runtime setup

1. Open this notebook in **Google Colab**.
2. Select **Runtime → Change runtime type → T4 GPU**.
3. Run all cells from top to bottom.
4. When asked, upload `training_assessment.zip` from this single project folder.
5. Approve the Google Drive mount. Training outputs are saved to Drive so completed experiments survive a Colab disconnect.

**Important:** Do not train on the dev or test files. The notebook uses them only for scoring, model selection, error analysis and final inference. The rules-in-prompt baseline is reference-only; all fine-tuning and final test predictions use the original fixed system/user messages.


In [ ]:
# In Colab: Runtime -> Change runtime type -> T4 GPU.
import os, sys, json, csv, math, time, gc, re, random, shutil, zipfile, subprocess, textwrap
from pathlib import Path
from collections import Counter

assert 'google.colab' in sys.modules, 'Open this notebook in Google Colab for the T4 GPU workflow.'
!nvidia-smi

Sat Nov 14 08:15:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%pip -q install "transformers==4.51.3" "peft==0.15.2" "accelerate==1.6.0" "bitsandbytes==0.45.5" "datasets==3.5.0" "pandas>=2.0" "scikit-learn>=1.3"

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    Trainer, TrainingArguments, DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from torch.utils.data import WeightedRandomSampler

assert torch.cuda.is_available(), 'No CUDA GPU detected. Select a T4 GPU runtime.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
SEED = 42
MAX_NEW_TOKENS = 200
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/VIMA3YA_NimbusPay_Assessment')
OUTPUT_DIR = DRIVE_ROOT / 'outputs'
WORK_DIR = DRIVE_ROOT / 'work'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent output folder:', OUTPUT_DIR)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GPU: Tesla T4
PyTorch: 2.14.1+cu130
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent output folder: /content/drive/MyDrive/VIMA3YA_NimbusPay_Assessment/outputs


## 1. Upload and inspect the assessment pack

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select the supplied training_assessment.zip
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_files, 'Please upload the original assessment ZIP.'
assessment_zip = Path('/content') / zip_files[0]
extract_dir = Path('/content/assessment_pack')
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(assessment_zip) as zf:
    zf.extractall(extract_dir)
train_paths = list(extract_dir.rglob('train.jsonl'))
assert len(train_paths) == 1, f'Expected one train.jsonl; found {train_paths}'
pack_root = train_paths[0].parent.parent
DATA_DIR = train_paths[0].parent
SCHEMA_PATH = pack_root / 'SCHEMA.md'
SCORE_PATH = pack_root / 'score.py'
assert SCHEMA_PATH.exists() and SCORE_PATH.exists()
print('Pack root:', pack_root)
print('Train:', DATA_DIR / 'train.jsonl')
print('Dev:', DATA_DIR / 'dev.jsonl')
print('Test inputs:', DATA_DIR / 'test_inputs.jsonl')

Saving training_assessment.zip to training_assessment (2).zip
Pack root: /content/assessment_pack/candidate_pack
Train: /content/assessment_pack/candidate_pack/data/train.jsonl
Dev: /content/assessment_pack/candidate_pack/data/dev.jsonl
Test inputs: /content/assessment_pack/candidate_pack/data/test_inputs.jsonl


## 2. Audit and clean training data (Part A)

The complete cleaning implementation is embedded in this notebook and is written to the Colab work directory before execution. It reads **only** `train.jsonl` and applies the supplied `SCHEMA.md` rules. Unparseable assistant targets are dropped; legacy keys, date/ID formatting, priority and `needs_human` labels are repaired with a row-level log.

In [ ]:
PREPARE_DATA_SOURCE = '#!/usr/bin/env python3\n"""Audit and clean NimbusPay ticket-triage training rows.\n\nThis utility is embedded in notebook.ipynb. The notebook writes it to the\nColab work directory and executes it automatically. It reads only the supplied\ntraining split and SCHEMA.md rules; dev/test are never read by this script.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport re\nfrom collections import Counter\nfrom copy import deepcopy\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Any\n\nKEYS = [\n    "category", "priority", "amount_paise", "txn_date",\n    "txn_id", "channel", "language", "needs_human",\n]\nCATEGORIES = {\n    "payment_failed", "refund", "fraud", "account_access",\n    "kyc", "offers", "other",\n}\nCHANNELS = {"upi", "card", "netbanking", "wallet"}\n\nLEGAL_PATTERN = re.compile(\n    r"\\b(rbi|ombudsman|consumer court|lawyer|legal notice|legal action)\\b",\n    flags=re.IGNORECASE,\n)\n\n# Explicit repeat-contact wording from SCHEMA.md and variants observed in train.\nREPEAT_PHRASES = (\n    "third time", "3rd time", "second time", "2nd time", "fourth time", "4th time",\n    "second complaint", "third complaint", "fourth complaint",\n    "again and again", "over and over", "baar baar", "baar-baar",\n    "teesri baar", "doosri baar", "pehle bhi complaint kar chuka",\n    "pehle bhi complaint kar chuki", "pehle bhi complaint kiya",\n    "contacted support about this before",\n    "contacted support about this issue before",\n    "already written 2 times", "already written 3 times", "already written 4 times",\n    "already written twice", "already complained", "keep raising this again and again",\n)\n\n\ndef load_jsonl(path: Path) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    with path.open(encoding="utf-8") as f:\n        for line_no, line in enumerate(f, start=1):\n            if not line.strip():\n                continue\n            try:\n                rows.append(json.loads(line))\n            except json.JSONDecodeError as exc:\n                raise ValueError(f"Outer JSONL error at {path}:{line_no}: {exc}") from exc\n    return rows\n\n\ndef get_user_text(row: dict[str, Any]) -> str:\n    for message in row.get("messages", []):\n        if message.get("role") == "user":\n            return str(message.get("content", ""))\n    return ""\n\n\ndef compute_priority(label: dict[str, Any], user_text: str) -> str:\n    """Apply the priority rules in SCHEMA.md in their specified order."""\n    category = label.get("category")\n    amount = label.get("amount_paise")\n    p1 = (\n        category == "fraud"\n        or (isinstance(amount, int) and not isinstance(amount, bool) and amount >= 5_000_000)\n        or bool(LEGAL_PATTERN.search(user_text))\n    )\n    if p1:\n        return "P1"\n\n    p2 = (\n        category == "account_access"\n        or (\n            category in {"payment_failed", "refund"}\n            and isinstance(amount, int)\n            and not isinstance(amount, bool)\n            and amount >= 500_000\n        )\n    )\n    return "P2" if p2 else "P3"\n\n\ndef requires_human(priority: str, user_text: str) -> bool:\n    lowered = user_text.casefold()\n    repeat_contact = any(phrase in lowered for phrase in REPEAT_PHRASES)\n    return priority == "P1" or repeat_contact\n\n\ndef normalize_date(value: Any) -> Any:\n    if value is None:\n        return None\n    if not isinstance(value, str):\n        return value\n    value = value.strip()\n    if re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", value):\n        # Validate actual calendar date too.\n        datetime.strptime(value, "%Y-%m-%d")\n        return value\n    if re.fullmatch(r"\\d{2}/\\d{2}/\\d{4}", value):\n        return datetime.strptime(value, "%d/%m/%Y").strftime("%Y-%m-%d")\n    if re.fullmatch(r"\\d{2}-\\d{2}-\\d{2}", value):\n        return datetime.strptime(value, "%d-%m-%y").strftime("%Y-%m-%d")\n    raise ValueError(f"Unrecognized txn_date format: {value!r}")\n\n\ndef normalize_txn_id(value: Any) -> Any:\n    if value is None:\n        return None\n    if not isinstance(value, str):\n        raise ValueError(f"txn_id must be string or null, got {type(value).__name__}")\n    compact = re.sub(r"[^A-Za-z0-9]", "", value).upper()\n    if not re.fullmatch(r"NP\\d{10}", compact):\n        raise ValueError(f"Cannot normalize txn_id safely: {value!r}")\n    return compact\n\n\ndef validate_label(label: dict[str, Any], row_id: str) -> None:\n    if set(label) != set(KEYS):\n        raise ValueError(f"{row_id}: wrong keys after cleaning: {sorted(label)}")\n    if label["category"] not in CATEGORIES:\n        raise ValueError(f"{row_id}: invalid category {label[\'category\']!r}")\n    if label["priority"] not in {"P1", "P2", "P3"}:\n        raise ValueError(f"{row_id}: invalid priority {label[\'priority\']!r}")\n    amount = label["amount_paise"]\n    if amount is not None and (not isinstance(amount, int) or isinstance(amount, bool)):\n        raise ValueError(f"{row_id}: amount_paise must be integer or null")\n    date = label["txn_date"]\n    if date is not None:\n        if not isinstance(date, str) or not re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", date):\n            raise ValueError(f"{row_id}: txn_date is not ISO YYYY-MM-DD")\n        datetime.strptime(date, "%Y-%m-%d")\n    txn_id = label["txn_id"]\n    if txn_id is not None and (not isinstance(txn_id, str) or not re.fullmatch(r"NP\\d{10}", txn_id)):\n        raise ValueError(f"{row_id}: invalid txn_id {txn_id!r}")\n    if label["channel"] not in CHANNELS | {None}:\n        raise ValueError(f"{row_id}: invalid channel {label[\'channel\']!r}")\n    if label["language"] not in {"en", "hi-en"}:\n        raise ValueError(f"{row_id}: invalid language {label[\'language\']!r}")\n    if not isinstance(label["needs_human"], bool):\n        raise ValueError(f"{row_id}: needs_human must be boolean")\n    if label["category"] in {"account_access", "kyc", "other"}:\n        for key in ("amount_paise", "txn_date", "txn_id", "channel"):\n            if label[key] is not None:\n                raise ValueError(f"{row_id}: {key} must be null for {label[\'category\']}")\n\n\ndef clean_rows(rows: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], list[dict[str, str]], dict[str, Any]]:\n    cleaned: list[dict[str, Any]] = []\n    log: list[dict[str, str]] = []\n    dropped = 0\n    fix_reason_counts: Counter[str] = Counter()\n    fix_field_counts: Counter[str] = Counter()\n    malformed_ids: list[str] = []\n    raw_category_counts: Counter[str] = Counter()\n    cleaned_category_counts: Counter[str] = Counter()\n    priority_mismatches = 0\n    needs_human_mismatches = 0\n\n    for row in rows:\n        row_id = str(row.get("id", "UNKNOWN"))\n        messages = row.get("messages", [])\n        if len(messages) < 3 or messages[-1].get("role") != "assistant":\n            raise ValueError(f"{row_id}: expected system/user/assistant messages")\n        raw_target = messages[-1].get("content", "")\n        try:\n            label = json.loads(raw_target)\n        except (json.JSONDecodeError, TypeError):\n            dropped += 1\n            malformed_ids.append(row_id)\n            log.append({\n                "id": row_id,\n                "action": "drop",\n                "reason": "Assistant target is truncated or malformed JSON; the complete target cannot be reconstructed from the row alone.",\n            })\n            continue\n        if not isinstance(label, dict):\n            dropped += 1\n            malformed_ids.append(row_id)\n            log.append({"id": row_id, "action": "drop", "reason": "Assistant target is not a JSON object."})\n            continue\n\n        original = deepcopy(label)\n        reasons: list[str] = []\n        user_text = get_user_text(row)\n\n        # A noisy subset uses `prio` and low/medium/high rather than the schema key.\n        if "prio" in label and "priority" not in label:\n            raw_priority = label.pop("prio")\n            mapped = {"low": "P3", "medium": "P2", "high": "P1"}.get(str(raw_priority).casefold())\n            if mapped is None:\n                raise ValueError(f"{row_id}: cannot map legacy prio value {raw_priority!r}")\n            label["priority"] = mapped\n            reasons.append("renamed legacy prio field to priority and mapped low/medium/high")\n            fix_field_counts["prio->priority"] += 1\n\n        # Canonicalize only values already supplied in the label; never hallucinate fields.\n        normalized_date = normalize_date(label.get("txn_date"))\n        if normalized_date != label.get("txn_date"):\n            label["txn_date"] = normalized_date\n            reasons.append("normalized txn_date to YYYY-MM-DD")\n            fix_field_counts["txn_date"] += 1\n\n        normalized_id = normalize_txn_id(label.get("txn_id"))\n        if normalized_id != label.get("txn_id"):\n            label["txn_id"] = normalized_id\n            reasons.append("normalized txn_id to uppercase NP plus 10 digits")\n            fix_field_counts["txn_id"] += 1\n\n        raw_cat = label.get("category")\n        raw_category_counts[str(raw_cat)] += 1\n\n        expected_priority = compute_priority(label, user_text)\n        if label.get("priority") != expected_priority:\n            priority_mismatches += 1\n            label["priority"] = expected_priority\n            reasons.append("recomputed priority using the ordered SCHEMA.md rules")\n            fix_field_counts["priority"] += 1\n\n        expected_human = requires_human(expected_priority, user_text)\n        if label.get("needs_human") != expected_human:\n            needs_human_mismatches += 1\n            label["needs_human"] = expected_human\n            reasons.append("recomputed needs_human from P1 status or explicit repeat-contact wording")\n            fix_field_counts["needs_human"] += 1\n\n        # Do not silently coerce any other unsupported schema value.\n        validate_label(label, row_id)\n        row_copy = deepcopy(row)\n        row_copy["messages"][-1]["content"] = json.dumps(label, ensure_ascii=False)\n        cleaned.append(row_copy)\n        cleaned_category_counts[str(label["category"])] += 1\n        if reasons:\n            log.append({"id": row_id, "action": "fix", "reason": "; ".join(reasons) + "."})\n            for reason in reasons:\n                fix_reason_counts[reason] += 1\n\n    summary = {\n        "input_rows": len(rows),\n        "clean_rows": len(cleaned),\n        "dropped_rows": dropped,\n        "fixed_rows": sum(item["action"] == "fix" for item in log),\n        "cleaning_log_rows": len(log),\n        "malformed_assistant_json_ids": malformed_ids,\n        "legacy_prio_rows": fix_field_counts["prio->priority"],\n        "noncanonical_date_rows": fix_field_counts["txn_date"],\n        "noncanonical_txn_id_rows": fix_field_counts["txn_id"],\n        "priority_mismatches_repaired": priority_mismatches,\n        "needs_human_mismatches_repaired": needs_human_mismatches,\n        "field_fix_counts": dict(fix_field_counts),\n        "original_category_counts_among_parseable_rows": dict(raw_category_counts),\n        "clean_category_counts": dict(cleaned_category_counts),\n        "fix_reason_counts": dict(fix_reason_counts),\n    }\n    return cleaned, log, summary\n\n\ndef write_jsonl(rows: list[dict[str, Any]], path: Path) -> None:\n    with path.open("w", encoding="utf-8") as f:\n        for row in rows:\n            f.write(json.dumps(row, ensure_ascii=False) + "\\n")\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train", required=True, type=Path, help="Path to supplied train.jsonl")\n    parser.add_argument("--out-dir", required=True, type=Path)\n    args = parser.parse_args()\n\n    rows = load_jsonl(args.train)\n    cleaned, log, summary = clean_rows(rows)\n    args.out_dir.mkdir(parents=True, exist_ok=True)\n    write_jsonl(cleaned, args.out_dir / "clean_train.jsonl")\n    with (args.out_dir / "cleaning_log.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=["id", "action", "reason"])\n        writer.writeheader()\n        writer.writerows(log)\n    (args.out_dir / "audit_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'
prepare_script = WORK_DIR / 'prepare_data.py'
prepare_script.write_text(PREPARE_DATA_SOURCE, encoding='utf-8')
subprocess.run([
    sys.executable, str(prepare_script),
    '--train', str(DATA_DIR / 'train.jsonl'),
    '--out-dir', str(WORK_DIR / 'audit'),
], check=True)
CLEAN_TRAIN_PATH = WORK_DIR / 'audit' / 'clean_train.jsonl'
CLEANING_LOG_PATH = WORK_DIR / 'audit' / 'cleaning_log.csv'
AUDIT_SUMMARY = json.loads((WORK_DIR / 'audit' / 'audit_summary.json').read_text())
print('\nAudit summary:')
print(json.dumps(AUDIT_SUMMARY, indent=2))



Audit summary:
{
  "input_rows": 1445,
  "clean_rows": 1420,
  "dropped_rows": 25,
  "fixed_rows": 157,
  "cleaning_log_rows": 182,
  "malformed_assistant_json_ids": [
    "tr-00060",
    "tr-00095",
    "tr-00125",
    "tr-00172",
    "tr-00267",
    "tr-00357",
    "tr-00478",
    "tr-00497",
    "tr-00558",
    "tr-00605",
    "tr-00674",
    "tr-00735",
    "tr-00765",
    "tr-00809",
    "tr-00824",
    "tr-00846",
    "tr-00958",
    "tr-00960",
    "tr-00975",
    "tr-00978",
    "tr-00979",
    "tr-00980",
    "tr-01070",
    "tr-01087",
    "tr-01199"
  ],
  "legacy_prio_rows": 70,
  "noncanonical_date_rows": 30,
  "noncanonical_txn_id_rows": 20,
  "priority_mismatches_repaired": 65,
  "needs_human_mismatches_repaired": 4,
  "field_fix_counts": {
    "priority": 65,
    "prio->priority": 70,
    "txn_date": 30,
    "txn_id": 20,
    "needs_human": 4
  },
  "original_category_counts_among_parseable_rows": {
    "payment_failed": 414,
    "account_access": 183,
    "refund": 31

In [ ]:
def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

raw_train_all = read_jsonl(DATA_DIR / 'train.jsonl')
# The raw-vs-clean comparison uses every row whose original assistant target is parseable.
raw_train = []
for row in raw_train_all:
    try:
        obj = json.loads(row['messages'][-1]['content'])
        if isinstance(obj, dict):
            raw_train.append(row)
    except (json.JSONDecodeError, TypeError):
        pass
clean_train = read_jsonl(CLEAN_TRAIN_PATH)
dev_rows = read_jsonl(DATA_DIR / 'dev.jsonl')
test_rows = read_jsonl(DATA_DIR / 'test_inputs.jsonl')

assert len(raw_train) == 1420
assert len(clean_train) == 1420
assert len(dev_rows) == 200
assert len(test_rows) == 400
assert not set(r['id'] for r in clean_train) & set(r['id'] for r in dev_rows)
assert not set(r['id'] for r in clean_train) & set(r['id'] for r in test_rows)
print(f'Parseable raw train: {len(raw_train)} | cleaned train: {len(clean_train)} | dev: {len(dev_rows)} | test: {len(test_rows)}')
display(pd.read_csv(CLEANING_LOG_PATH).head(12))

Parseable raw train: 1420 | cleaned train: 1420 | dev: 200 | test: 400


,id,action,reason
0,tr-00013,fix,recomputed priority using the ordered SCHEMA.m...
1,tr-00018,fix,recomputed priority using the ordered SCHEMA.m...
2,tr-00020,fix,renamed legacy prio field to priority and mapp...
3,tr-00023,fix,recomputed priority using the ordered SCHEMA.m...
4,tr-00045,fix,recomputed priority using the ordered SCHEMA.m...
5,tr-00049,fix,renamed legacy prio field to priority and mapp...
6,tr-00058,fix,renamed legacy prio field to priority and mapp...
7,tr-00059,fix,recomputed priority using the ordered SCHEMA.m...
8,tr-00060,drop,Assistant target is truncated or malformed JSO...
9,tr-00071,fix,normalized txn_id to uppercase NP plus 10 digits.


### Cleaning validation

In [ ]:
EXPECTED_KEYS = {'category','priority','amount_paise','txn_date','txn_id','channel','language','needs_human'}
for row in clean_train:
    label = json.loads(row['messages'][-1]['content'])
    assert set(label) == EXPECTED_KEYS, (row['id'], label.keys())
    assert label['priority'] in {'P1','P2','P3'}
    assert type(label['needs_human']) is bool
print('PASS: all retained training labels have exactly the 8 required keys and valid core types.')
print('Cleaning log:', CLEANING_LOG_PATH)


PASS: all retained training labels have exactly the 8 required keys and valid core types.
Cleaning log: /content/drive/MyDrive/VIMA3YA_NimbusPay_Assessment/work/audit/cleaning_log.csv


## 3. Load the official scorer

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('official_score', SCORE_PATH)
official_score = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official_score)

def score_predictions(gold_rows, pred_rows):
    result = official_score.score(gold_rows, pred_rows)
    return result['all'], result


## 4. Base-model baselines (Part B)

Baseline 1 uses the exact fixed system prompt supplied in each row. Baseline 2 adds the complete schema rules to the system message; it is a reference-only upper-bound prompt and **must not** be used for final fine-tuning evaluation or test inference.

In [ ]:
def make_generation_text(row, include_schema_rules=False, tokenizer=None):
    messages = [dict(m) for m in row['messages'][:2]]
    if include_schema_rules:
        fixed_system = messages[0]['content']
        schema_text = SCHEMA_PATH.read_text(encoding='utf-8')
        messages[0]['content'] = fixed_system + '\n\nREFERENCE BASELINE ONLY — TASK RULES:\n' + schema_text
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_predictions(model, tokenizer, rows, include_schema_rules=False, batch_size=4):
    tokenizer.padding_side = 'left'
    outputs = []
    model.eval()
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start+batch_size]
        prompts = [make_generation_text(r, include_schema_rules, tokenizer) for r in batch]
        encoded = tokenizer(prompts, return_tensors='pt', padding=True, truncation=False)
        device = next(model.parameters()).device
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        new_tokens = generated[:, encoded['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        # Contract: whitespace stripping only. Never repair JSON or fill missing fields.
        outputs.extend([text.strip() for text in texts])
    return [{'id': row['id'], 'output': output} for row, output in zip(rows, outputs)]

def load_fp16_base():
    tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map='auto'
    )
    model.config.use_cache = True
    return model, tok

base_model, tokenizer = load_fp16_base()
base_predictions = generate_predictions(base_model, tokenizer, dev_rows, include_schema_rules=False)
base_metrics, base_detail = score_predictions(dev_rows, base_predictions)
print('Baseline 1 — fixed prompt')
print(json.dumps(base_metrics, indent=2))

rules_predictions = generate_predictions(base_model, tokenizer, dev_rows, include_schema_rules=True)
rules_metrics, rules_detail = score_predictions(dev_rows, rules_predictions)
print('Baseline 2 — rules in prompt (reference only)')
print(json.dumps(rules_metrics, indent=2))

(OUTPUT_DIR / 'baseline_fixed_dev.jsonl').write_text(''.join(json.dumps(x, ensure_ascii=False)+'\n' for x in base_predictions), encoding='utf-8')
(OUTPUT_DIR / 'baseline_rules_dev.jsonl').write_text(''.join(json.dumps(x, ensure_ascii=False)+'\n' for x in rules_predictions), encoding='utf-8')
del base_model
gc.collect()
torch.cuda.empty_cache()


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/con

Baseline 1 — fixed prompt
{
  "n": 200,
  "json_valid": 0.0,
  "exact_match": 0.0,
  "mean_field_acc": 0.0,
  "fields": {
    "category": 0.0,
    "priority": 0.0,
    "amount_paise": 0.0,
    "txn_date": 0.0,
    "txn_id": 0.0,
    "channel": 0.0,
    "language": 0.0,
    "needs_human": 0.0
  }
}
Baseline 2 — rules in prompt (reference only)
{
  "n": 200,
  "json_valid": 0.14,
  "exact_match": 0.005,
  "mean_field_acc": 0.086875,
  "fields": {
    "category": 0.11,
    "priority": 0.035,
    "amount_paise": 0.115,
    "txn_date": 0.105,
    "txn_id": 0.12,
    "channel": 0.095,
    "language": 0.075,
    "needs_human": 0.04
  }
}


## 5. Sequence-length evidence and assistant-only tokenization (Part C)

The sequence-length percentile is measured on training rows only. Prompt tokens are masked with `-100`, so the loss is computed only on the assistant target. Truncation, if needed, removes the oldest prompt tokens and keeps the complete target.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def tokenized_length(row):
    return len(tokenizer.apply_chat_template(row['messages'], tokenize=True, add_generation_prompt=False))
train_lengths = [tokenized_length(r) for r in clean_train]
p50, p95, p99 = (int(pd.Series(train_lengths).quantile(q)) for q in [0.50, 0.95, 0.99])
MAX_SEQ_LEN = min(768, max(256, int(math.ceil(p95 / 8) * 8)))
print({'train_rows': len(clean_train), 'p50_tokens': p50, 'p95_tokens': p95, 'p99_tokens': p99, 'max_seq_len': MAX_SEQ_LEN})

def tokenize_supervised_rows(rows, tokenizer, max_seq_len):
    features = []
    for row in rows:
        messages = row['messages']
        prompt_ids = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)
        full_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError(f"Chat-template prefix mismatch for {row['id']}; inspect template before training.")
        target_len = len(full_ids) - len(prompt_ids)
        if target_len <= 0 or target_len > max_seq_len:
            raise ValueError(f"Invalid assistant target length for {row['id']}: {target_len}")
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        cut = max(0, len(full_ids) - max_seq_len)
        features.append({
            'input_ids': full_ids[cut:],
            'attention_mask': [1] * len(full_ids[cut:]),
            'labels': labels[cut:],
        })
    return features

clean_features = tokenize_supervised_rows(clean_train, tokenizer, MAX_SEQ_LEN)
raw_features = tokenize_supervised_rows(raw_train, tokenizer, MAX_SEQ_LEN)
assert len(clean_features) == len(clean_train)
assert all(any(x != -100 for x in item['labels']) for item in clean_features)
print('Assistant-only labels verified; features:', len(clean_features))

{'train_rows': 1420, 'p50_tokens': 168, 'p95_tokens': 226, 'p99_tokens': 1810, 'max_seq_len': 256}
Assistant-only labels verified; features: 1420


## 6. Training and experiment runner (Parts C and D)

Controlled comparisons implemented:

- **E1 — data quality:** raw parseable labels vs cleaned labels, both 4-bit QLoRA rank 8.
- **E2 — quantization:** cleaned 16-bit LoRA rank 8 vs the cleaned 4-bit QLoRA rank 8 result from E1.
- **E3 — adapter capacity:** cleaned QLoRA rank 8 vs rank 16.
- **E4 — sampling:** cleaned QLoRA with ordinary sampling vs category-balanced weighted sampling.

The common settings are held fixed: base model, chat template, max sequence length, epochs, optimizer, learning rate, batch size and seed. Each row in the resulting table includes measured dev metrics, GPU memory and training time.

In [ ]:
# Remove the incompatible installation
%pip uninstall -y bitsandbytes

# Install the latest available compatible release
%pip install --no-cache-dir --upgrade bitsandbytes

Found existing installation: bitsandbytes 0.45.5
Uninstalling bitsandbytes-0.45.5:
  Successfully uninstalled bitsandbytes-0.45.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 86.4 MB/s  0:00:00


In [ ]:
import torch
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Bitsandbytes:", bnb.__version__)

assert torch.cuda.is_available(), "GPU is not available"

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

# Test actual 4-bit NF4 computation
from bitsandbytes.nn import Linear4bit

layer = Linear4bit(
    16,
    16,
    bias=False,
    compute_dtype=torch.float16,
    quant_type="nf4"
).to("cuda")

x = torch.randn(2, 16, device="cuda", dtype=torch.float16)
y = layer(x)

print("NF4 test:", y.shape)
print("GPU quantization test PASSED")

PyTorch: 2.14.1+cu130
CUDA version: 13.0
CUDA available: True
Bitsandbytes: 0.50.2
GPU: Tesla T4
Compute capability: (7, 5)
NF4 test: torch.Size([2, 16])
GPU quantization test PASSED


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (16) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


In [13]:
def load_base_for_training(load_in_4bit):
    if load_in_4bit:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=quant_config, device_map='auto'
        )
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=torch.float16, device_map='auto'
        )
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.enable_input_require_grads()
    model.config.use_cache = False
    return model

class CategoryBalancedTrainer(Trainer):
    def __init__(self, *args, sample_weights=None, **kwargs):
        self.sample_weights = sample_weights
        super().__init__(*args, **kwargs)
    def _get_train_sampler(self, train_dataset=None):
        dataset = train_dataset if train_dataset is not None else self.train_dataset
        if self.sample_weights is None:
            return super()._get_train_sampler(dataset)
        return WeightedRandomSampler(
            weights=self.sample_weights,
            num_samples=len(dataset),
            replacement=True,
        )

def make_training_arguments(output_dir):
    return TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_ratio=0.03,
        lr_scheduler_type='cosine',
        optim='paged_adamw_8bit',
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_strategy='steps',
        logging_steps=10,
        save_strategy='no',
        report_to='none',
        dataloader_num_workers=2,
        dataloader_pin_memory=False,
        max_grad_norm=0.3,
        seed=SEED,
        data_seed=SEED,
        remove_unused_columns=True,
    )

def run_experiment(name, source_rows, load_in_4bit=True, rank=8, balanced=False):
    tokenizer.padding_side = 'right'  # causal-LM training uses right padding
    run_dir = OUTPUT_DIR / 'experiments' / name
    run_dir.mkdir(parents=True, exist_ok=True)
    features = tokenize_supervised_rows(source_rows, tokenizer, MAX_SEQ_LEN)
    ds = Dataset.from_list(features)
    sample_weights = None
    if balanced:
        categories = [json.loads(r['messages'][-1]['content']).get('category', 'other') for r in source_rows]
        counts = Counter(categories)
        sample_weights = [1.0 / counts[c] for c in categories]
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()
    model = load_base_for_training(load_in_4bit=load_in_4bit)
    lora_config = LoraConfig(
        task_type='CAUSAL_LM',
        r=rank,
        lora_alpha=16,  # held constant for the rank comparison
        lora_dropout=0.05,
        bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    assert trainable <= 50_000_000, f'{trainable:,} trainable parameters exceeds 50M'
    print(f'{name}: trainable={trainable:,} / total={total:,}')
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8,
        label_pad_token_id=-100,
        return_tensors='pt',
    )
    trainer_cls = CategoryBalancedTrainer if balanced else Trainer
    trainer_kwargs = {'sample_weights': sample_weights} if balanced else {}
    trainer = trainer_cls(
        model=model,
        args=make_training_arguments(run_dir / 'trainer_state'),
        train_dataset=ds,
        data_collator=collator,
        **trainer_kwargs,
    )
    start = time.perf_counter()  # exclude model download/load time from training minutes
    train_result = trainer.train()
    training_minutes = (time.perf_counter() - start) / 60.0
    peak_gpu_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    model.config.use_cache = True
    model.save_pretrained(run_dir)
    tokenizer.save_pretrained(run_dir)
    dev_predictions = generate_predictions(model, tokenizer, dev_rows, include_schema_rules=False)
    metrics, detail = score_predictions(dev_rows, dev_predictions)
    result = {
        'name': name,
        'data': 'raw' if source_rows is raw_train else 'clean',
        'load_in_4bit': bool(load_in_4bit),
        'rank': int(rank),
        'balanced_sampling': bool(balanced),
        'trainable_parameters': int(trainable),
        'dev_json_valid': metrics['json_valid'],
        'dev_exact_match': metrics['exact_match'],
        'dev_mean_field_accuracy': metrics['mean_field_acc'],
        'peak_gpu_memory_gb': peak_gpu_gb,
        'training_minutes': training_minutes,
        'adapter_dir': str(run_dir),
        'training_log': trainer.state.log_history,
        'dev_predictions': dev_predictions,
        'dev_detail': detail,
    }
    (run_dir / 'run_metrics.json').write_text(json.dumps({k:v for k,v in result.items() if k not in {'dev_predictions','dev_detail'}}, indent=2))
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return result

EXPERIMENTS = [
    {'name':'E1_raw_qlora_r8', 'source':'raw', 'load_in_4bit':True, 'rank':8, 'balanced':False},
    {'name':'E1_clean_qlora_r8', 'source':'clean', 'load_in_4bit':True, 'rank':8, 'balanced':False},
    {'name':'E2_clean_lora_fp16_r8', 'source':'clean', 'load_in_4bit':False, 'rank':8, 'balanced':False},
    {'name':'E3_clean_qlora_r16', 'source':'clean', 'load_in_4bit':True, 'rank':16, 'balanced':False},
    {'name':'E4_clean_qlora_r8_balanced', 'source':'clean', 'load_in_4bit':True, 'rank':8, 'balanced':True},
]

run_results = []
for exp in EXPERIMENTS:
    source = raw_train if exp['source'] == 'raw' else clean_train
    try:
        result = run_experiment(
            exp['name'], source,
            load_in_4bit=exp['load_in_4bit'],
            rank=exp['rank'],
            balanced=exp['balanced'],
        )
        run_results.append(result)
        print({k:result[k] for k in ['name','dev_exact_match','dev_mean_field_accuracy','peak_gpu_memory_gb','training_minutes']})
    except Exception as exc:
        # Preserve the failure honestly; never fabricate metrics for a run that did not finish.
        failure = {'name': exp['name'], 'status':'FAILED', 'error':repr(exc)}
        run_results.append(failure)
        (OUTPUT_DIR / f"{exp['name']}_failure.txt").write_text(repr(exc), encoding='utf-8')
        print('EXPERIMENT FAILED:', failure)
        gc.collect()
        torch.cuda.empty_cache()

successful_runs = [r for r in run_results if 'dev_exact_match' in r]
assert successful_runs, 'No fine-tuning experiment completed. Review the logged error files and adjust only if necessary.'
best_run = sorted(successful_runs, key=lambda x:(x['dev_exact_match'], x['dev_mean_field_accuracy'], x['dev_json_valid']), reverse=True)[0]
print('\nSelected best run by dev exact match, then mean field accuracy, then JSON validity:')
print({k:best_run[k] for k in ['name','load_in_4bit','rank','dev_exact_match','dev_mean_field_accuracy']})


def load_base_for_inference(load_in_4bit):
    if load_in_4bit:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=quant_config, device_map='auto'
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=torch.float16, device_map='auto'
        )
    model.config.use_cache = True
    return model


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


E1_raw_qlora_r8: trainable=9,232,384 / total=897,848,832


Step,Training Loss
10,1.551600
20,0.260800
30,0.106700
40,0.069000
50,0.063500
60,0.051200
70,0.049500
80,0.053800
90,0.066700
100,0.052700


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'name': 'E1_raw_qlora_r8', 'dev_exact_match': 0.745, 'dev_mean_field_accuracy': 0.956875, 'peak_gpu_memory_gb': 3.0042266845703125, 'training_minutes': 23.56465268346666}


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


... (16 outputs left out)

Step,Training Loss
10,1.570900
20,0.280400
30,0.081100
40,0.073300
50,0.052500
60,0.045200
70,0.045700
80,0.044400
90,0.035700
100,0.021400


Step,Training Loss
10,1.570900
20,0.280400
30,0.081100
40,0.073300
50,0.052500
60,0.045200
70,0.045700
80,0.044400
90,0.035700
100,0.021400


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'name': 'E4_clean_qlora_r8_balanced', 'dev_exact_match': 0.785, 'dev_mean_field_accuracy': 0.960625, 'peak_gpu_memory_gb': 3.0026397705078125, 'training_minutes': 24.425257748733355}

Selected best run by dev exact match, then mean field accuracy, then JSON validity:
{'name': 'E2_clean_lora_fp16_r8', 'load_in_4bit': False, 'rank': 8, 'dev_exact_match': 0.865, 'dev_mean_field_accuracy': 0.97375}


## 7. Experiment results and error analysis (Parts D and E)

In [14]:
table_rows = []
for r in run_results:
    if 'dev_exact_match' not in r:
        table_rows.append({'experiment':r['name'], 'status':'FAILED', 'dev_exact_match':None, 'dev_mean_field_accuracy':None, 'peak_gpu_memory_gb':None, 'training_minutes':None, 'notes':r.get('error','')})
    else:
        table_rows.append({
            'experiment':r['name'], 'status':'completed',
            'dev_exact_match':r['dev_exact_match'],
            'dev_mean_field_accuracy':r['dev_mean_field_accuracy'],
            'peak_gpu_memory_gb':r['peak_gpu_memory_gb'],
            'training_minutes':r['training_minutes'],
            'notes':f"data={r['data']}; 4bit={r['load_in_4bit']}; rank={r['rank']}; balanced={r['balanced_sampling']}; trainable={r['trainable_parameters']}",
        })
experiment_df = pd.DataFrame(table_rows)
display(experiment_df)
experiment_df.to_csv(OUTPUT_DIR / 'experiment_results.csv', index=False)

best_detail = best_run['dev_detail']['all'              ]
field_scores = pd.DataFrame([{'field':k,'accuracy':v} for k,v in best_detail['fields'].items()]).sort_values('accuracy')
category_scores = []
for key, value in best_run['dev_detail'].items():
    if key.startswith('category='):
        category_scores.append({'category':key.split('=',1)[1], 'n':value['n'], 'exact_match':value['exact_match'], 'mean_field_accuracy':value['mean_field_acc']})
category_df = pd.DataFrame(category_scores).sort_values('exact_match')
print('Weakest fields:')
display(field_scores)
print('Weakest categories:')
display(category_df)

pred_map = {p['id']:p['output'] for p in best_run['dev_predictions']}
errors = []
for row in dev_rows:
    gold = json.loads(row['messages'][-1]['content'])
    raw_output = pred_map[row['id']]
    try:
        pred = json.loads(raw_output.strip())
    except Exception:
        pred = None
    if pred is None or set(pred) != EXPECTED_KEYS or any(type(pred.get(k)) is not type(gold[k]) or pred.get(k) != gold[k] for k in EXPECTED_KEYS):
        wrong = [k for k in EXPECTED_KEYS if pred is None or type(pred.get(k)) is not type(gold[k]) or pred.get(k) != gold[k]]
        errors.append({
            'id': row['id'],
            'ticket': row['messages'][1]['content'],
            'gold_json': json.dumps(gold, ensure_ascii=False),
            'raw_model_output': raw_output,
            'wrong_fields': ','.join(sorted(wrong)),
            'explanation_to_write': '',
        })
error_df = pd.DataFrame(errors)
error_df.head(10).to_csv(OUTPUT_DIR / 'dev_error_analysis_first10.csv', index=False)
print(f'Incorrect dev rows: {len(errors)} / {len(dev_rows)}. First 10 saved for human explanation.')

summary = {
    'model_id': MODEL_ID,
    'selected_run': {k:best_run[k] for k in ['name','data','load_in_4bit','rank','balanced_sampling','trainable_parameters','dev_json_valid','dev_exact_match','dev_mean_field_accuracy','peak_gpu_memory_gb','training_minutes']},
    'baseline_fixed_prompt': base_metrics,
    'baseline_rules_prompt_reference_only': rules_metrics,
    'weakest_fields': field_scores.to_dict(orient='records'),
    'weakest_categories': category_df.to_dict(orient='records'),
    'incorrect_dev_rows': len(errors),
    'total_dev_rows': len(dev_rows),
}
(OUTPUT_DIR / 'report_results.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved report_results.json')

,experiment,status,dev_exact_match,dev_mean_field_accuracy,peak_gpu_memory_gb,training_minutes,notes
0,E1_raw_qlora_r8,completed,0.745,0.956875,3.004227,23.564653,data=raw; 4bit=True; rank=8; balanced=False; t...
1,E1_clean_qlora_r8,completed,0.835,0.971875,3.002640,23.891404,data=clean; 4bit=True; rank=8; balanced=False;...
2,E2_clean_lora_fp16_r8,completed,0.865,0.973750,3.898509,18.145970,data=clean; 4bit=False; rank=8; balanced=False...
3,E3_clean_qlora_r16,completed,0.815,0.968125,3.065765,24.298446,data=clean; 4bit=True; rank=16; balanced=False...
4,E4_clean_qlora_r8_balanced,completed,0.785,0.960625,3.002640,24.425258,data=clean; 4bit=True; rank=8; balanced=True; ...


Weakest fields:


,field,accuracy
1,priority,0.900
7,needs_human,0.940
0,category,0.970
2,amount_paise,0.985
3,txn_date,0.995
4,txn_id,1.000
5,channel,1.000
6,language,1.000


Weakest categories:


,category,n,exact_match,mean_field_accuracy
2,refund,55,0.763636,0.956818
4,fraud,15,0.800000,0.941667
3,payment_failed,53,0.830189,0.966981
0,other,16,0.937500,0.992188
6,account_access,19,0.947368,0.993421
1,kyc,24,1.000000,1.000000
5,offers,18,1.000000,1.000000


Incorrect dev rows: 27 / 200. First 10 saved for human explanation.
Saved report_results.json


## 8. Save selected adapter and generate hidden-test predictions

This section reloads the selected base-model precision and the saved best adapter. Final inference uses only the original fixed `system` and `user` messages. The only output post-processing is `.strip()`; invalid JSON is retained as generated and will score as incorrect rather than being repaired.

In [15]:
best_adapter_dir = Path(best_run['adapter_dir'])
assert (best_adapter_dir / 'adapter_config.json').exists()
final_adapter_dir = OUTPUT_DIR / 'adapter'
if final_adapter_dir.exists():
    shutil.rmtree(final_adapter_dir)
shutil.copytree(best_adapter_dir, final_adapter_dir)

final_base = load_base_for_inference(load_in_4bit=best_run['load_in_4bit'])
final_model = PeftModel.from_pretrained(final_base, str(final_adapter_dir), is_trainable=False)
final_model.config.use_cache = True

# Required hidden test: 400 unlabelled rows, no schema rules or dev labels.
test_predictions = generate_predictions(final_model, tokenizer, test_rows, include_schema_rules=False, batch_size=4)
assert len(test_predictions) == len(test_rows) == 400
pred_path = OUTPUT_DIR / 'predictions_test.jsonl'
pred_path.write_text(''.join(json.dumps(p, ensure_ascii=False)+'\n' for p in test_predictions), encoding='utf-8')

# Deterministic reproducibility check on a fixed random sample of test IDs.
sample_rows = random.Random(2026).sample(test_rows, 20)
reproduced = generate_predictions(final_model, tokenizer, sample_rows, include_schema_rules=False, batch_size=4)
original_map = {p['id']:p['output'] for p in test_predictions}
reproduced_map = {p['id']:p['output'] for p in reproduced}
repro_matches = sum(reproduced_map[k] == original_map[k] for k in reproduced_map)
assert repro_matches == len(sample_rows), f'Reproducibility mismatch: {repro_matches}/{len(sample_rows)}'
print(f'Reproducibility check: {repro_matches}/{len(sample_rows)} identical outputs.')
print('Predictions:', pred_path)

# Validate line count, id order and generated JSON/schema without changing the outputs.
assert [p['id'] for p in test_predictions] == [r['id'] for r in test_rows]
valid_json = 0
for p in test_predictions:
    try:
        obj = json.loads(p['output'].strip())
        if isinstance(obj, dict) and set(obj) == EXPECTED_KEYS:
            valid_json += 1
    except json.JSONDecodeError:
        pass
print(f'JSON objects with the exact 8-key structure: {valid_json}/400')

# Save experiment/audit metadata alongside final files, but do not include source data.
shutil.copy2(CLEANING_LOG_PATH, OUTPUT_DIR / 'cleaning_log.csv')
shutil.copy2(WORK_DIR / 'audit' / 'audit_summary.json', OUTPUT_DIR / 'audit_summary.json')
print('Final adapter files:', [p.name for p in final_adapter_dir.iterdir()])


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Reproducibility check: 20/20 identical outputs.
Predictions: /content/drive/MyDrive/VIMA3YA_NimbusPay_Assessment/outputs/predictions_test.jsonl
JSON objects with the exact 8-key structure: 400/400
Final adapter files: ['trainer_state', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'run_metrics.json']


## 9. Package the generated artifacts

In [16]:
# Contains adapter, test predictions, cleaning log, audit/metrics and error-analysis file.
# It does not contain the supplied train/dev/test data or the base model.
archive_path = shutil.make_archive('/content/nimbuspay_submission_artifacts', 'zip', root_dir=OUTPUT_DIR)
print('Artifact bundle:', archive_path)
from google.colab import files
files.download(archive_path)

Artifact bundle: /content/nimbuspay_submission_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Before submitting

1. Review `experiment_results.csv`; include only completed runs and explain any failed run honestly.
2. Open `dev_error_analysis_first10.csv` and write a specific explanation for each of at least 10 failures. If fewer than 10 dev rows fail, inspect additional field-level near-misses and report that fact accurately.
3. Update `report.md` and `README.md` with the metrics from `report_results.json`. Do not copy placeholder or illustrative numbers.
4. Copy the generated `adapter/` and `predictions_test.jsonl` into the private GitHub repository.
5. Keep the assessment data and base model out of GitHub. Do not push after the deadline.
6. Perform the live defence using the notebook and explain the decisions, failures and limitations.